In [2]:
"""
PIPELINE STAGE: Smart Temporal Consolidation & Automated Schema Normalization
PERIOD: January 2020 - December 2024
LIBRARIES: os, pandas

1. OBJECTIVE:
   Intelligently merge two longitudinal datasets (2020-2021 and 2021-2024) into a 
   single, continuous time-series database. This script ensures structural alignment by 
   enforcing the first dataset's schema as the master reference, neutralizing header 
   discrepancies between two distinct processing pipelines.

2. FILE ARCHITECTURE:
   - Input 1 (Master Schema): os.path.join("..", "..", "processed_data", "steps", "03_2020_2021_final.xlsx")
   - Input 2 (Append Target): os.path.join("..", "..", "processed_data", "steps", "06_2021_2024_final.xlsx")
   - Output (Unified Data): os.path.join("..", "..", "processed_data", "steps", "07_final_energy_consolidation.xlsx")

3. SMART SCHEMA NORMALIZATION (Critical Logic):
   A. Header Synchronization: To prevent data fragmentation during vertical concatenation 
      (appending rows), the script treats Input 1 as the definitive "Master Table".
   B. Automated Column Mapping: It validates structural compatibility by checking column 
      counts. If they match exactly, Input 2's headers are systematically overwritten with 
      the Master Headers to guarantee 100% alignment without data shifting.
   C. Safeguard Mechanism: Implements a validation check to warn the user if column 
      dimensions are inconsistent, averting structural corruption prior to integration.

4. DATA INTEGRATION & INDEXING:
   - Method: Vertical concatenation across temporal boundaries using pd.concat.
   - Index Management: Resets indices (ignore_index=True) to generate a clean, continuous 
     primary key sequence for the unified time-series dataset.

5. AUDIT & ERROR RESILIENCE:
   - Real-time logging of the ingestion process, distinct row counts per file, and the 
     final consolidated record count.
   - Comprehensive try-except encapsulation to capture I/O exceptions, ensuring 
     transparent and user-friendly debugging.
"""


import pandas as pd
import os

def excel_birlestirme_operasyonu():
    # 1. Klasör ve dosya yollarını tanımla
    dosya1_yol = os.path.join("..", "..", "processed_data", "steps", "03_2020_2021_final.xlsx")
    dosya2_yol = os.path.join("..","..", "processed_data", "steps", "06_2021_2024_final.xlsx")
    cikti_yol = os.path.join("..", "..", "processed_data", "steps", "07_final_energy_consolidation.xlsx")

    try:
        # 2. İlk dosyayı oku (Başlıkları buradan alacağız)
        df1 = pd.read_excel(dosya1_yol)
        print(f"İlk dosya okundu: {dosya1_yol} ({len(df1)} satır)")

        # 3. İkinci dosyayı oku
        # header=0 diyerek mevcut başlıkları okuyoruz ama sonra ezeceğiz
        df2 = pd.read_excel(dosya2_yol)
        
        # İkinci dosyanın sütun isimlerini, ilk dosyanın isimleriyle birebir değiştir
        # Bu işlem başlıkları değil, sadece altındaki veriyi df1 formatına sokar
        if len(df1.columns) == len(df2.columns):
            df2.columns = df1.columns
            print(f"İkinci dosya başlıkları normalize edildi: {dosya2_yol} ({len(df2)} satır)")
        else:
            print("UYARI: Dosyaların sütun sayıları eşit değil! Veri kayması olabilir.")

        # 4. İki tabloyu alt alta birleştir
        # ignore_index=True ile satır numaralarını 0'dan başlayacak şekilde yeniden düzenle
        final_df = pd.concat([df1, df2], ignore_index=True)

        # 5. Sonucu yeni dosya olarak kaydet
        final_df.to_excel(cikti_yol, index=False)
        
        print("-" * 30)
        print(f"İŞLEM BAŞARILI!")
        print(f"Toplam satır sayısı: {len(final_df)}")
        print(f"Kayıt yeri: {cikti_yol}")

    except Exception as e:
        print(f"HATA OLUŞTU: {e}")

if __name__ == "__main__":
    excel_birlestirme_operasyonu()

İlk dosya okundu: ..\..\processed_data\steps\03_2020_2021_final.xlsx (1782 satır)
İkinci dosya başlıkları normalize edildi: ..\..\processed_data\steps\06_2021_2024_final.xlsx (3078 satır)
------------------------------
İŞLEM BAŞARILI!
Toplam satır sayısı: 4860
Kayıt yeri: ..\..\processed_data\steps\07_final_energy_consolidation.xlsx
